In [ ]:
%pip install -q bitsandbytes datasets transformers accelerate peft

import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Läuft auf:", device)


In [ ]:
# %%
model_name = "MBZUAI/LaMini-GPT-1.5B"  # Reasoning-fähiges 1.5B-Modell
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Lade Basismodell in 4-Bit ...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Falls noch kein pad_token_id vorhanden, auf eos_token_id setzen
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
# ==========================
# 3) Vorbereitung für LoRA-Finetuning
# ==========================
model = prepare_model_for_kbit_training(base_model)
model.gradient_checkpointing_enable()  # Memory sparen

# LoRA-Konfiguration:
# - r=16 (Größe der Low-Rank-Matrizen)
# - lora_dropout=0.1 (etwas höher, da Datensatz klein -> Overfitting reduzieren)
# - target_modules: an die GPT-Architektur anpassen (hier: q_proj, k_proj, etc.)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "fc_in", "fc_out"],
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# z.B. "Yonekin/nius_reasoning"
dataset_name = "Yonekin/nius_reasoning"
ds = load_dataset("json", data_files="https://huggingface.co/datasets/Yonekin/nius_reasoning/resolve/main/qa_sheet_cleaned2.json")

# Künstliche Aufteilung in Train/Val: 80% zu 20%
ds = ds["train"].train_test_split(test_size=0.2, seed=42)
train_ds = ds["train"]
val_ds = ds["test"]

print("Train set size:", len(train_ds))
print("Val set size:", len(val_ds))

In [ ]:
# ==========================
# 5) Tokenize-Funktion mit Truncation ( Notwendig?)
# ==========================
# Wir wollen Reasoning + Artikel gleich wichtig nehmen.
# Format => "Reasoning:\n<reasoning>\n\nArticle:\n<output>"

MAX_LENGTH = 1024  # Wir nehmen 1024 Tokens an
# Bei langen Einträgen: wir truncaten.

def tokenize_fn(example):
   
    prompt = f"PROMPT:\n{example['input']}\n\nREASONING:\n{example['reasoning']}\n\nARTICLE:\n{example['output']}"
    
    # Tokenisierung mit Truncation
    enc = tokenizer(prompt, 
                    max_length=MAX_LENGTH, 
                    truncation=True, 
                    return_tensors="pt")
    
    input_ids = enc["input_ids"][0].tolist()  # [0] da batch=1
    attention_mask = enc["attention_mask"][0].tolist()

    # Alle Tokens sind Label-Tokens -> wir wollen Reasoning & Artikel outputten
    # Falls du "input" nicht in die Loss einbeziehen willst, müsstest du Prompt-Teil mit -100 maskieren.
    # Hier machen wir es einfach: wir lassen ALLES trainierbar, 
    # also das Modell soll den gesamten Text reproduzieren. 
    # Im Normalfall würde man Prompt tokens => -100, 
    # reasoning + artikel => labels. 
    # Ich demonstriere letzeres (Prompt maskieren):
    prompt_enc = tokenizer(f"PROMPT:\n{example['input']}", truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    prompt_len = len(prompt_enc["input_ids"][0])
    
    # Alles bis prompt_len => -100
    labels = [-100]*prompt_len + input_ids[prompt_len:]
    # Wir müssen schauen, ob das ge-truncated wurde => min check
    labels = labels[:MAX_LENGTH]
    input_ids = input_ids[:MAX_LENGTH]
    attention_mask = attention_mask[:MAX_LENGTH]
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

train_ds = train_ds.map(tokenize_fn, remove_columns=train_ds.column_names)
val_ds = val_ds.map(tokenize_fn, remove_columns=val_ds.column_names)


In [ ]:
# ==========================
# 6) Trainingskonfiguration
# ==========================
# - Kleine Lernrate z.B. 5e-5, um Overfitting zu reduzieren.
# - gradient_accumulation_steps=16 => simulierte Batch 16 * per_device_train_batch_size
# - Wir überwachen die Validierungsperformance (perplexity).
# - Cosine Scheduler, Warmup ratio 0.03, um Smooth Start/End zu haben.

training_args = TrainingArguments(
    output_dir="./lora-lamini-1.5b-checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=16,   # => effektive Batch-Größe 32
    num_train_epochs=4,              # wir probieren 4 Epochen
    learning_rate=5e-5,              # etwas niedriger für den kleinen Datensatz
    fp16=True,
    logging_steps=50,
    evaluation_strategy="steps",
    eval_steps=200,                  # alle 200 Steps evaluieren wir
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",
    report_to="none"
)

In [ ]:
# ==========================
# 7) Trainer + Perplexity-Metrik
# ==========================
import math
def compute_metrics(eval_preds):
    # Standard: eval_preds => (logits, labels)
    logits, labels = eval_preds
    # => logits: [batch_size, seq_length, vocab_size]
    # => labels: [batch_size, seq_length]
    # berechnen wir cross entropy loss => perplexity
    predictions = torch.from_numpy(logits)
    label_ids = torch.from_numpy(labels)
    
    # wir maskieren positions= -100 in labels
    loss_fct = torch.nn.CrossEntropyLoss(ignore_index=-100)
    shift_logits = predictions[..., :-1, :].contiguous()
    shift_labels = label_ids[..., 1:].contiguous()
    # => z.B. next token prediction
    
    loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    perplexity = math.exp(loss.item())
    return {"perplexity": perplexity}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=lambda data: tokenizer.pad(data, return_tensors="pt", padding=True),
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
# ==========================
# 9) Inferenz-Test
# ==========================
model.eval()
test_prompt = "PROMPT:\nGib mir bitte eine kurze Zusammenfassung der heutigen Wirtschaftsnachrichten.\n\nREASONING:\n"

input_ids = tokenizer(test_prompt, return_tensors="pt")["input_ids"].to(device)
with torch.no_grad():
    gen_output = model.generate(
        input_ids=input_ids,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
text_out = tokenizer.decode(gen_output[0], skip_special_tokens=True)
print("================================")
print("GENERATED TEXT:")
print(text_out[len(test_prompt):])  # reasoning + article

In [ ]:
trainer.model.save_pretrained("./lamini-1.5b-lora-adapter")
tokenizer.save_pretrained("./lamini-1.5b-lora-adapter")